# 원본과 PQ의 동일 FIQA 보정 비교

**질문:** 품질 보정의 이득이 원본에서도 같은가, PQ 압축 후 추가로 커지는가?

Origin × Global-safe/5-bin/Continuous와 PQ m128/m64/m32 × 같은 세 방법을 비교합니다.
같은 query·gallery·신원 분할·FIQA·fit/safety 규칙을 검증한 후 실행합니다.
기존 00/01/02와 완료 결과를 보존하는 별도 실험입니다.

기본 **EXECUTE=False는 읽기 전용 입력 검증**입니다. 첫 설정 셀만 수정한 뒤
**Kernel Restart → Run All** 하세요. EXECUTE=True에서는 입력 파생 자료와 결과를
자동 저장하며, 완료된 원본 검색 shard와 run/seed 작업은 해시 검증 후 재사용합니다.

In [ ]:
# 0. 모든 사용자 설정 — 변경 후 Kernel Restart → Run All
from pathlib import Path
import sys
from copy import deepcopy

PROJECT_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (p / "research").is_dir() and (p / ".git").exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("프로젝트 안에서 실행하세요.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.experiments.calibration_matrix import DEFAULT_RUN_MATRIX
RUN_MATRIX = deepcopy(DEFAULT_RUN_MATRIX)  # 명시적 기존 12개 run; 자동 latest 선택 없음

DATASETS = ("lfw", "rfw_custom", "survface")
MODELS = ("arcface", "adaface", "magface", "edgeface")
PQ_PROFILES = ("pq_512_m128_b8", "pq_512_m64_b8", "pq_512_m32_b8")
FIQA_VARIANT = "L"
CALIBRATION_ROOT = PROJECT_ROOT / "results/calibration"
OUTPUT_ROOT = CALIBRATION_ROOT / "origin_vs_pq"

EXECUTE = False
PARTITION_SEEDS = (*range(19), 8972)
ORIGIN_SHARD_SIZE = 4096

# 기존 PQ 적합 계수 재사용: source·입력·fit 설정·과학 코드·runtime 검증 후 허용.
# 불일치하면 중단합니다. 의도적으로 새 설정을 쓰려면 False로 바꾸어 새로 적합하세요.
REUSE_PQ_MODELS = True
SOURCE_REPORT_DIR = CALIBRATION_ROOT / "matrix/reports/matrix-report-d690567c6f4f7e7ad44ef19e"
FIT_SETTINGS = dict(
    target_fpirs=(.01, .05, .10, .20, .30),
    safety_fraction=.30, minimum_group_non_mated=100, shrinkage_strength=200.,
    knot_quantiles=(1/3, 2/3), smoothing=.01, ridge=.001, max_iterations=2000,
    margin_slope_cap=.95,  # 기존 quality-only 모델 기록과 일치; FIQA 단조성 제약 아님
    resamples=2000, bootstrap_seed=8972,
    diagnostic_fpir_grid=(0., .001, .005, .01, .02, .05, .1, .2, .3, .5, 1.),
)

# 아래는 표시할 표만 선택하며 실험 범위/방법 선택에 사용하지 않습니다.
DISPLAY_DATASET = "lfw"
DISPLAY_MODEL = "arcface"
DISPLAY_TARGET_FPIR = .10
DISPLAY_DIAGNOSTIC_FPIR = .10


## 1. 입력과 실행 범위 확인

기본 범위는 3개 데이터셋 × 4모델 × (원본 + 3개 PQ) = **48개 표현 조건**입니다.
48 × 3방법 × 5목표 × 20분할 = **14,400개 운영 성능 행**을 저장합니다.
범위를 바꾸면 아래에서 실제 행 수를 다시 계산합니다.

원본 test의 최대 점수·정답 점수·정답 순위는 기존 검색 ledger를 재사용합니다.
원본 calibration만 동일한 enrollment 규칙으로 정확 cosine 검색합니다.
이 검색은 기존 NumPy **CPU** 구현이며, FR/FIQA 추론·Grad-CAM·압축기 학습·DB 작업은 없습니다.
원본과 PQ의 점수 공간이 다르므로 각각 보정하며 원본 threshold를 ADC로 복사하지 않습니다.

PQ condition score 또는 FIQA가 없거나 손상되면 정확한 경로와 함께 중단합니다.
누락 입력은 기존 입력 준비 절차에서 만든 뒤 재개하세요.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display
from research.experiments.origin_pq_inputs import inspect_origin_pq_experiment
from research.experiments.origin_vs_pq_calibration import (
    OriginPQSettings, VerifiedPQModels, run_origin_pq_campaign,
)

settings = OriginPQSettings(**FIT_SETTINGS)
settings.validate()
plan = inspect_origin_pq_experiment(
    PROJECT_ROOT, run_matrix=RUN_MATRIX, datasets=DATASETS, models=MODELS,
    profiles=PQ_PROFILES, variant=FIQA_VARIANT, calibration_root=CALIBRATION_ROOT,
)
source_report = VerifiedPQModels(SOURCE_REPORT_DIR) if REUSE_PQ_MODELS else None
representation_count = len(plan) + plan.source_run_id.nunique()
expected_rows = representation_count * 3 * len(settings.target_fpirs) * len(PARTITION_SEEDS)
print("Python:", sys.executable)
print(f"입력 검증 완료: {plan.source_run_id.nunique()}개 run, "
      f"{representation_count}개 표현 조건, {expected_rows:,}개 운영 성능 행")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 100):
    display(plan[["dataset_id", "model", "compression_profile", "source_run_id", "test_rows", "ready"]])
print("출력:", OUTPUT_ROOT)
print("실험 실행:", EXECUTE, "/ 기존 PQ 모델 재사용:", REUSE_PQ_MODELS)


## 2. 실험 실행과 중단 후 재개

각 run에서 원본 calibration 검색을 shard 단위로 저장하고, 각 seed의 전체 표현 비교를
하나의 완료 작업으로 저장합니다. 같은 설정으로 재실행하면 완료 자료를 검증하고 건너뜁니다.
중간에 끝나지 않은 seed의 적합/평가는 다시 수행할 수 있습니다.

REUSE_PQ_MODELS=True일 때 기존 PQ **모델 계수**를 검증 후 불러옵니다.
FPIR/TPIR, 실패 분해, paired 비교, 원본과의 상호작용, 진단 곡선은 새 비교를 위해 계산합니다.
원본에는 같은 보정 함수를 새로 적합합니다. Test를 바꿔도 적합 계수와 fit/safety 분할은 바뀌지 않습니다.

전역 FPIR 목표를 충족해도 품질 구간별 FPIR가 같거나 보장되는 것은 아닙니다.
동일 test를 공유한 20분할은 calibration 분할 민감도이며 독립 재현 20회가 아닙니다.

In [ ]:
result = None
if EXECUTE:
    status = display(Markdown("입력 준비 중…"), display_id=True)
    def show_progress(event):
        status.update(Markdown("진행: " + " · ".join(f"{k}={v}" for k, v in event.items())))
    result = run_origin_pq_campaign(
        plan, OUTPUT_ROOT, partition_seeds=PARTITION_SEEDS, settings=settings,
        pq_model_report=SOURCE_REPORT_DIR if REUSE_PQ_MODELS else None,
        shard_size=ORIGIN_SHARD_SIZE, progress=show_progress,
    )
    status.update(Markdown("완료: " + str(result["report_dir"])))
else:
    print("사전 검증만 완료했습니다. 실험하려면 첫 셀의 EXECUTE=True로 변경하세요.")


## 3. 운영 성능과 실패 원인

Threshold는 calibration에서 고정합니다. method_summary.csv에는 반올림 전 FPIR,
목표 충족, 성공 수·분모, Wilson CI, TPIR의 신원 단위 bootstrap CI가 있습니다.
아래는 선택 조건의 분할별 min/median/max와 목표 충족 횟수입니다.

등록 질의는 **성공 / 정답 Top-K 밖의 순위 실패 / Top-K 안의 threshold 실패**로 나눕니다.
세 개의 수를 합하면 등록 질의 수입니다. rank_k_ceiling은 threshold로 회복 가능한 범위의
진단 상한이며, 같은 FPIR에서 실제 달성할 수 있다는 뜻은 아닙니다.

In [ ]:
if result is not None:
    view = result["split_summary"]
    view = view.loc[view.dataset_id.eq(DISPLAY_DATASET) & view.model.eq(DISPLAY_MODEL)
                    & view.target_fpir.eq(DISPLAY_TARGET_FPIR)]
    display(view[["compression_profile", "method", "split_count", "target_met_split_count",
                  "fpir_min", "fpir_median", "fpir_max", "tpir_min", "tpir_median", "tpir_max",
                  "rank_failure_rate", "threshold_failure_median"]])
    print("전체 조건의 원시 성능과 실패 수:", result["report_dir"] / "method_summary.csv")


## 4. 압축과 보정의 관계

paired_comparisons.csv는 같은 표현의 방법 차이와 같은 방법의 PQ−원본 차이를 담습니다.
interactions.csv의 정의는 다음과 같습니다.

**I = (PQ의 FIQA−Global) − (원본의 FIQA−Global)**

운영 표의 I는 **같은 목표 FPIR**에서의 차분입니다. 실제 FPIR는 다를 수 있으므로 네 실제
FPIR와 목표 충족을 함께 확인합니다. TPIR 차분의 CI는 같은 등록 신원을 함께 재표집합니다.
인과효과·압축 특이 알고리즘·그룹 FPIR 보장을 자동으로 뜻하지 않습니다.

In [ ]:
if result is not None:
    view = result["interactions"]
    view = view.loc[view.dataset_id.eq(DISPLAY_DATASET) & view.model.eq(DISPLAY_MODEL)
                    & view.target_fpir.eq(DISPLAY_TARGET_FPIR)]
    # 여러 seed의 범위는 기술통계이며 하나의 합친 신뢰구간이 아닙니다.
    display(view.groupby(["compression_profile", "method"]).agg(
        split_count=("partition_seed", "count"), all_four_met=("all_four_target_met", "sum"),
        pq_gain_median=("pq_gain", "median"), origin_gain_median=("origin_gain", "median"),
        interaction_min=("interaction", "min"), interaction_median=("interaction", "median"),
        interaction_max=("interaction", "max")))
    print("각 seed의 실제 FPIR와 paired CI:", result["report_dir"] / "interactions.csv")


## 5. 같은 FPIR에서의 사후 진단

한 번 적합한 함수의 score − threshold(FIQA)를 고정하고 scalar cutoff를 이동해 곡선을 만듭니다.
같은 FPIR 좌표의 TPIR는 경험적 곡선을 선형 보간한 값입니다. 동점 구간은 무작위 threshold
혼합에 해당할 수 있으며 **운영 가능한 단일 threshold를 선택한 결과가 아닙니다**.

진단 곡선과 진단 I에는 현재 CI가 없습니다. fitted_target_fpir가 다른 모델들의 점을
하나의 ROC처럼 이어 붙이지 않습니다. 아래는 설정한 하나의 적합 목표와 하나의 진단 FPIR만
표시합니다. 운영 결과는 앞의 표를 사용하세요.

In [ ]:
if result is not None:
    view = result["diagnostic_interactions"]
    view = view.loc[view.dataset_id.eq(DISPLAY_DATASET) & view.model.eq(DISPLAY_MODEL)
                    & view.fitted_target_fpir.eq(DISPLAY_TARGET_FPIR)
                    & view.diagnostic_fpir.eq(DISPLAY_DIAGNOSTIC_FPIR)]
    display(view.groupby(["compression_profile", "method"]).agg(
        split_count=("partition_seed", "count"), pq_gain_median=("pq_gain", "median"),
        origin_gain_median=("origin_gain", "median"), interaction_median=("interaction", "median")))
    print("전체 진단 곡선:", result["report_dir"] / "diagnostic_curves.csv")
    print("모델 계수·재사용 출처:", result["report_dir"] / "models.csv")
    print("fit/safety 분할 증거:", result["report_dir"] / "partition_inventory.csv")
    print("보고서:", result["report_dir"])


## 해석과 다음 단계

- PQ와 원본에서 같은 정도로 개선되면 일반 품질 보정의 효용에 가깝습니다.
  압축 조건에서 추가 이득이 남는지 운영 표와 동일 FPIR 진단을 함께 확인합니다.
- 좋은 조건만 골라 전체 경향으로 해석하지 않습니다. 목표 실패와 음의 차이도 저장합니다.
- models.csv에서 fitting_reused와 출처를 확인할 수 있습니다. 원본 캐시와 모든 결과는
  별도 results/calibration/origin_vs_pq/ 아래에 저장하며 기존 완료 자료는 수정하지 않습니다.
- 이 노트북은 구현 검증과 실험 수행을 구분합니다. 전체 행렬 Run All을 수행하기 전에는
  48조건·20분할의 최종 성능 결과가 완성됐다고 보고하지 않습니다.
